# SHF — online fusion, multi-run evaluation

Classical + quantum branches fused online (`OnlineFusionModel`). Setup lives in `src/utils/fusion_experiment.py`; the multi-run machinery in `src/utils/multirun.py` — both shared with `run-shf.py`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'requirements.txt').exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root (looked for requirements.txt)")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.utils import fusion_experiment as fx
from src.utils.multirun import (
    run_multi_seed,
    evaluate_checkpoints,
    restore_quantum_rand_params,
)

device = fx.get_device()
print(f"Device: {device}")
print(f"Project root: {project_root}")

In [ ]:
# One config = one model. The same dict drives run-shf.py, so a notebook result is
# reproducible from the CLI: `python run-shf.py --only shallow`.
CONFIG = fx.resolve({
    'dataset': 'BreastMNIST',
    'classical_model': 'shallow',       # 'shallow' | 'resnet'
    'quantum_model': 'quantum',         # 'quantum' | 'extended_quantum'

    # run-shf.py uses freeze=True/True for resnet and False/False for shallow.
    'freeze_classical': False,
    'freeze_quantum': False,

    'batch_size': 16,
    'image_size': 28,
    'num_epochs': 10,
    'feature_dim': 128,
    'num_classes': 2,

    'lr_classical': 1e-4,
    'lr_quantum': 1e-2,
    'lr_fusion': 1e-3,

    'n_qubits': 4,
    'n_layers': 1,
    'patch_size': 2,
})

N_RUNS = 5
REPORTS_DIR = fx.reports_dir(CONFIG, 'shf')
RUN_TAG = fx.run_tag(CONFIG)
TITLE_PREFIX = f"SHF {CONFIG['classical_model']}+{CONFIG['quantum_model']}"

print(fx.describe(CONFIG))

In [ ]:
train_loader, test_loader, val_loader = fx.build_dataloaders(CONFIG)

print(f"{CONFIG['dataset']} — train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)} batches")
print(f"Reports dir: {REPORTS_DIR}")
print(f"Run tag:     {RUN_TAG}")

## Multi-run evaluation

Trains the model above **N_RUNS** times from scratch (seeds `0..N_RUNS-1`) and reports **mean ± std** plus a **95% confidence interval** (Student's t) for every test metric — including **average precision (AP)** — together with per-run and aggregated confusion matrices, **precision-recall curves** and ROC curves.

Everything is written under `reports/shf/` (see `reports/README.md`). The identical sweep runs headless via `python run-shf.py`.

In [ ]:
result = run_multi_seed(
    fx.make_build_fn(CONFIG, device),
    train_loader, val_loader, test_loader, device,
    reports_dir=REPORTS_DIR,
    run_tag=RUN_TAG,
    n_runs=N_RUNS,
    epochs=CONFIG['num_epochs'],
    num_classes=CONFIG['num_classes'],
    hyperparams=fx.hyperparams(CONFIG),
    report_meta=fx.report_meta(CONFIG),
    title_prefix=TITLE_PREFIX,
)

## Test-only evaluation (independent)

Reloads the `best_*.pth` checkpoints written by the cell above and regenerates every metric and figure — **no training**. Safe to run on its own after the setup cells, or headless via `python run-shf.py --test-only`.

In [ ]:
# Independent of the training cell above: reloads best_*.pth from REPORTS_DIR/multi_run
# and regenerates every metric and figure. Run the setup cells (imports, CONFIG,
# dataloaders) and then this one, in a fresh kernel, to re-test a finished experiment.
test_result = evaluate_checkpoints(
    fx.make_build_fn(CONFIG, device),
    test_loader, device,
    reports_dir=REPORTS_DIR,
    run_tag=RUN_TAG,
    n_runs=N_RUNS,
    num_classes=CONFIG['num_classes'],
    hyperparams=fx.hyperparams(CONFIG),
    report_meta=fx.report_meta(CONFIG),
    title_prefix=TITLE_PREFIX,
)

## UMAP of the learned representations

Projects the classical, quantum and fused feature spaces of one run's best checkpoint down to 2D.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import umap

UMAP_RUN = 1  # which seed's checkpoint to visualize

# Same seed the run was trained with — the non-trainable quantum circuit is drawn at
# build time and never enters the state_dict, so it has to be reproduced, not loaded.
torch.manual_seed(UMAP_RUN - 1)
np.random.seed(UMAP_RUN - 1)

umap_model = fx.build_model(CONFIG, device)
ckpt = os.path.join(REPORTS_DIR, "multi_run", f"best_{RUN_TAG}_run{UMAP_RUN}.pth")
restore_quantum_rand_params(umap_model, ckpt)   # exact circuit, from the .npy sidecar
umap_model.load_state_dict(torch.load(ckpt, map_location=device))
umap_model.eval()
print(f"Loaded {ckpt}")

branches = {"Classical": [], "Quantum": [], "Fused": []}
labels = []
with torch.no_grad():
    for images, batch_labels in test_loader:
        images = images.to(device)
        branches["Classical"].append(umap_model.classical_model.extract_features(images).flatten(1).cpu().numpy())
        branches["Quantum"].append(umap_model.quantum_model.extract_features(images).flatten(1).cpu().numpy())
        branches["Fused"].append(umap_model.extract_features(images).cpu().numpy())
        labels.append(np.asarray(batch_labels).squeeze().reshape(-1))

labels = np.concatenate(labels)
branches = {name: np.concatenate(feats, axis=0) for name, feats in branches.items()}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, feats) in zip(axes, branches.items()):
    embedding = umap.UMAP(n_components=2, random_state=42, n_neighbors=15,
                          min_dist=0.1).fit_transform(feats)
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=labels,
                         cmap="coolwarm", alpha=0.6, s=30)
    ax.set_title(f"UMAP — {name} features {feats.shape}")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2"); ax.grid(alpha=0.3)
fig.colorbar(scatter, ax=axes, label="Class", fraction=0.02, pad=0.02)

umap_path = os.path.join(REPORTS_DIR, "multi_run", f"umap_run{UMAP_RUN}.png")
fig.savefig(umap_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ UMAP saved at {umap_path}")